# 配位环境分析

以简单局域配位描述符分析训练集与生成结构的配位环境。
运行前设置数据目录和输出目录。


In [ ]:
import os
import glob
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

from pymatgen.core import Composition, Element, Structure
from matminer.featurizers.site.fingerprint import *
from matminer.featurizers.site.bonding import *
from matminer.featurizers.site.misc import *

import warnings
warnings.filterwarnings("ignore")

In [ ]:
anion_center_elements_list = [
    "B", "C", "N", "Si", "P", "S",
]

halogen_elements_list = [
    "F", "Cl", "Br", "I",
]

elements_list = [
    # "H", "He",
    "Li", "Be", "B", "C", "N", "O", "F",
    "Na", "Mg", "Al", "Si", "P", "S", "Cl",
    "K", "Ca", "Sc", "Ti", "Zn", "Ga", "Ge", "As", "Se", "Br",
    "Rb", "Sr", "Y", "Zr", "Nb", "Mo", "Ag", "Cd", "In", "Sn", "Sb", "Te", "I",
    "Cs", "Ba", "Hf", "Ta", "W", "Au", "Hg", "Tl", "Pb", "Bi", "Po", "At",
]

In [ ]:
sources = {
    "dataset": {"dataset_name": "5sets_data", "color": (237, 125, 49), "label": "MP Dataset"},
    "bcn_40": {"dataset_name": "bcn_40_gens", "color": (164, 192, 227), "label": "bcn_40"},
    "bp_40": {"dataset_name": "bp_40_gens", "color": (122, 169, 218), "label": "bp_40"},
    "csi_40": {"dataset_name": "csi_40_gens", "color": (86, 148, 203), "label": "csi_40"},
    "np_40": {"dataset_name": "np_40_gens", "color": (76, 132, 182), "label": "np_40"},
    "sps_30": {"dataset_name": "sps_30_gens", "color": (65, 113, 156), "label": "sps_30"},
}
work_dir = "C:/Users/13701/Desktop/gm_cifs/cdvae/v32/to_plot/"

In [ ]:
def get_fingerprints(
        input_structure: Structure,
        featurizer,
        elements_list: list = anion_center_elements_list,
):
    stats = ["max", "min", "mean", "std"]
    feature_size = None
    all_features = []
    for element in elements_list:
        site_indices = [
            idx for idx, site in enumerate(input_structure) if site.species_string == element
        ]
        if not site_indices:
            if feature_size is None:
                feature_size = len(featurizer.featurize(input_structure, 0))
            zero_feature = np.zeros(len(stats) * feature_size)
            all_features.append(zero_feature)
            continue

        element_features = np.array([
            featurizer.featurize(input_structure, idx) for idx in site_indices
        ])
        if feature_size is None:
            feature_size = element_features.shape[1]
        element_stats = []
        for stat in stats:
            stat_func = getattr(np, stat)
            element_stats.append(stat_func(element_features, axis=0))
        all_features.append(np.concatenate(element_stats))
    return np.concatenate(all_features)


def get_fingerprints_cn(
        input_structure: Structure,
        featurizer,
        elements_list: list = anion_center_elements_list,
) -> np.ndarray:
    feature_size = None
    all_features = []

    for element in elements_list:
        site_indices = [
            idx for idx, site in enumerate(input_structure) if site.species_string == element
        ]

        if not site_indices:
            if feature_size is None:
                feature_size = len(featurizer.featurize(input_structure, 0))
            zero_feature = np.zeros(2 * feature_size)
            all_features.append(zero_feature)
            continue

        element_features = np.array([
            featurizer.featurize(input_structure, idx) for idx in site_indices
        ])

        if feature_size is None:
            feature_size = element_features.shape[1]

        coordination_numbers = np.array([
            len(input_structure.get_neighbors(input_structure[idx], 3.0)) for idx in site_indices
        ])
        max_idx = np.argmax(coordination_numbers)
        min_idx = np.argmin(coordination_numbers)

        if len(site_indices) == 1:
            max_feature = element_features[max_idx]
            min_feature = element_features[max_idx]
        else:
            max_feature = element_features[max_idx]
            min_feature = element_features[min_idx]

        combined_features = np.concatenate([max_feature, min_feature])
        all_features.append(combined_features)

    return np.concatenate(all_features)


def get_simple_features(
        input_structure: Structure,
        elements_list: list = anion_center_elements_list,
        nn_method=None,
) -> np.ndarray:
    nn_method = CrystalNN() if nn_method is None else nn_method
    coord_number = CoordinationNumber(nn=nn_method)
    mean_bond_length = AverageBondLength(method=nn_method)
    mean_bond_angle = AverageBondAngle(method=nn_method)

    stats = ["max", "min", "mean", "std"]
    num_features = 3
    all_features = []

    for element in elements_list:
        site_indices = [
            idx for idx, site in enumerate(input_structure) if site.species_string == element
        ]

        if not site_indices:
            all_features.append(np.zeros(len(stats) * num_features))
            continue

        features = []
        for idx in site_indices:
            coord = coord_number.featurize(input_structure, idx)[0]
            bond_length = mean_bond_length.featurize(input_structure, idx)[0]
            bond_angle = mean_bond_angle.featurize(input_structure, idx)[0]
            features.append([coord, bond_length, bond_angle])

        features = np.array(features)

        element_stats = []
        for i in range(num_features):
            for stat in stats:
                stat_func = getattr(np, f"nan{stat}")
                element_stats.append(stat_func(features[:, i]))
        all_features.append(np.array(element_stats))

    return np.concatenate(all_features)

In [ ]:
filenames_all, fingerprints_all, labels_all = [], [], []

# featurizer = AGNIFingerprints(cutoff=5)
# featurizer = ChemEnvSiteFingerprint.from_preset("simple")
# featurizer = OPSiteFingerprint()
# featurizer = VoronoiFingerprint(cutoff=5.0)
# featurizer = BondOrientationalParameter()
# featurizer = CrystalNNFingerprint.from_preset("ops")

import time
start = time.time()
error_filenames = []
for source_name, source_data in sources.items():
    input_cif_files = glob.glob(os.path.join(work_dir, source_data["dataset_name"], "*.cif"))
    for input_cif_file in input_cif_files:
        input_cif_filename = os.path.basename(input_cif_file).split('.')[0]
        try:
            # fingerprints = get_fingerprints_cn(
            #     input_structure=Structure.from_file(input_cif_file),
            #     featurizer=featurizer,
            # )
            fingerprints = get_simple_features(
                input_structure=Structure.from_file(input_cif_file),
                # nn_method=VoronoiNN(cutoff=5.0),
            )
            if np.isnan(fingerprints).any():
                raise ValueError(f"NaN detected in fingerprints for {input_cif_filename}")
            filenames_all.append(input_cif_filename)
            fingerprints_all.append(fingerprints)
            labels_all.append(source_name)
        except Exception as e:
            error_filenames.append(input_cif_filename)
            print(f"Error processing {input_cif_filename}: {e}")
        if len(fingerprints_all) % 1000 == 0:
            time1 = time.time()
            print(f"Processed {len(fingerprints_all)} files, took {time1 - start:.2f} seconds")
            start = time.time()
print(len(filenames_all))
print(error_filenames)

In [ ]:
filenames_all_in = np.array(filenames_all)
descriptors_in = np.array(fingerprints_all)
print(filenames_all_in.shape)
print(descriptors_in.shape)

In [ ]:
for i in range(descriptors_in.shape[0]):
    for j in range(descriptors_in.shape[1]):
        if np.isnan(descriptors_in[i, j]):
            print(f"NaN in {filenames_all_in[i]} {i} {j}")

In [ ]:
tsne = TSNE(n_components=2, random_state=42, learning_rate=200.0, init="pca")
tsne_results = tsne.fit_transform(descriptors_in)
print(tsne_results.shape)

In [ ]:
from sklearn.preprocessing import MinMaxScaler
max_value = 95.
scaler = MinMaxScaler(feature_range=(-max_value, max_value))
tsne_results_scaled = scaler.fit_transform(tsne_results)

In [ ]:
import random
random.seed(42)
plt.rcParams["font.family"] = "Calibri"
plt.figure(figsize=(16, 16))
tsne_results_in = tsne_results_scaled
all_indices = []
for source_name, source_data in sources.items():
    # if source_name in ["dataset"]:
    if source_name:
        indices = [i for i, label in enumerate(labels_all) if label == source_name]
        if source_name == "dataset":
            indices = random.sample(indices, 2000)
            print(indices[-5:])
            all_indices += indices
            color_tmp = "orange"
        else:
            indices = random.sample(indices, 500)
            all_indices += indices
            color_tmp = "lightblue"
        plt.scatter(
            tsne_results_in[indices, 0],
            tsne_results_in[indices, 1],
            # color=[c/255 for c in source_data["color"]],
            color=color_tmp,
            # edgecolors="black",
            # linewidths=5,
            # facecolors="none",
            s=150,
            label=source_data["label"],
            alpha=0.8,  # transparency
        )

# plt.title("t-SNE Visualization", fontsize=32)
plt.xlim(-100, 100)
plt.ylim(-100, 100)
plt.xlabel(r"t-SNE$_1$", fontsize=40)
# plt.ylabel(r"t-SNE$_2$", fontsize=40)
# plt.tick_params(axis="x", which='major', labelsize=32, width=1)
plt.xticks([-100, -75, -50, -25, 0, 25, 50, 75, 100], fontsize=32)
# plt.yticks([-75, -50, -25, 0, 25, 50, 75, 100], fontsize=32)
plt.yticks([], fontsize=32)
plt.legend(fontsize=26, loc="upper right", ncol=2)  # 图例
# plt.grid(True)
plt.show()

In [ ]:
csv_file = "./f4b_plot.csv"
datarows = np.concatenate((filenames_all_in.reshape(-1, 1), tsne_results_scaled), axis=1)
datarows_to_plot = datarows[all_indices, :]
import csv
with open(csv_file, "w", newline="") as c_file:
    writer = csv.writer(c_file)
    writer.writerows(datarows_to_plot)